# 05 — CCHS 2019-2020: Diabetes

**Dataset:** Canadian Community Health Survey (CCHS), 2019-2020 Annual Component,
Public Use Microdata File (PUMF). Statistics Canada, distributed via
[Borealis Data](https://borealisdata.ca/dataset.xhtml?persistentId=doi:10.5683/SP3/ZVCGBK)
under the Statistics Canada Open Licence.

Restricted to respondents aged 18+ (n ~ 99,153).

**Outcome (`diabetes`):** derived from `CCC_095` ("Has diabetes"). Coded 1 if "Yes",
0 if "No"; "valid skip" / "don't know" / "refusal" / "not stated" rows are dropped.

**Protected attribute (`DHH_SEX`):** "Male" / "Female" (sex at birth).

**Covariate blocks for the case-mix waterfall:**
- `demographics`: age (group midpoint), education level, household income level,
  province (one-hot). CCHS does not collect a race/ethnicity variable comparable to
  the US surveys, so province is the closest available geographic/socioeconomic
  stratifier.
- `comorbidities`: high blood pressure, high cholesterol, overweight/obese (BMI class)
- `behavioral`: current smoker, moderate-to-vigorous physical activity minutes/week
- `access`: has a regular health care provider

This notebook runs the shared `run_fairness_analysis` pipeline (see `src/pipeline.py`)
on this dataset, exactly as for datasets 1-4, so results are directly comparable.
See `scripts/fetch_cchs.py` for the (manual download +) extraction steps used to
produce `data/cchs_2019_2020_subset.csv` from the full PUMF.


In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
RESULTS_DIR = ROOT / "results"
sys.path.insert(0, str(ROOT))

import os
import pandas as pd
import matplotlib.pyplot as plt

from src.datasets import load_cchs
from src.pipeline import run_fairness_analysis
from src import figures as figs


In [ ]:
d = load_cchs()
df = d["df"]

print(d["name"])
print("shape:", df.shape)
print("feature columns:", len(d["feature_cols"]))
print()
for val, lbl in d["group_labels"].items():
    sub = df[df[d["group_col"]] == val]
    n_pos = sub[d["target_col"]].sum()
    print(f"{lbl}: n={len(sub)}, positives={n_pos}, prevalence={n_pos / len(sub):.4f}")


## Run the fairness pipeline

As with datasets 2-4, we use `threshold="prevalence"`: diabetes prevalence in this
sample (~10%) is far from 50%, so a fixed 0.5 decision threshold would classify almost
no one as positive. Using the training prevalence as the threshold gives a more
balanced operating point.


In [ ]:
result = run_fairness_analysis(
    df=df,
    feature_cols=d["feature_cols"],
    target_col=d["target_col"],
    group_col=d["group_col"],
    group_a_value="Male",
    group_b_value="Female",
    group_labels=d["group_labels"],
    threshold="prevalence",
    n_boot=1000,
    covariate_blocks=d["covariate_blocks"],
    # Corrected path: preprocessing is fitted on training rows only,
    # and (where a clustering identifier exists) the split, selection
    # CV, calibration folds and bootstrap are patient-grouped.
    cluster_ids=None,
    preprocess_spec=d["preprocess_spec"],
)


## Interpretation

- **Raw gaps:** Men have a notably higher diabetes prevalence in this sample (11.7% vs
  8.5% in the test set, gap +0.032). The model's PPV is higher for men (+0.049) and NPV
  is higher for women (-0.018), both statistically significant.
- **Equal-opportunity / FNR gap is significant and replicates the cross-dataset
  pattern**: men have lower sensitivity (0.804 vs 0.852) and correspondingly higher FNR
  (+0.049, 95% CI excludes 0). This is the same direction seen in datasets 1-3 (CDC
  Diabetes, Diabetes-130, BRFSS) -- men are more likely to have their diabetes "missed"
  by the model than women, even though men have the higher underlying prevalence.
  Specificity/FPR show essentially no gap (~0.0005, not significant).
- **Prevalence adjustment:** Holding sensitivity/specificity fixed and recomputing
  PPV/NPV/predicted-positive-rate at a common prevalence attenuates the PPV gap by
  ~119% (from +0.049 to -0.009, which actually *flips sign* and remains significant in
  the opposite direction). The NPV gap attenuates by ~57% but remains significant in
  the same direction; the predicted-positive-rate gap was not significant raw and
  remains non-significant (and flips sign) after adjustment.
- **The sensitivity/FNR gap is NOT addressed by prevalence adjustment** (prevalence
  adjustment only touches PPV/NPV/predicted-positive-rate). This is exactly the kind of
  residual gap the project's hypothesis flags as the strongest candidate for genuine
  inequity: men's diabetes cases are disproportionately missed by the model, a pattern
  that has now replicated across four of five datasets (the exception, NHANES, had a
  much smaller sample and wide, non-significant CIs).
- **Calibration adjustment:** Under equal-sensitivity thresholds (matching the pooled
  sensitivity by adjusting each group's threshold), the equal-opportunity gap nearly
  vanishes (-0.0485 -> +0.0061) and the FNR gap correspondingly nearly vanishes too.
  This shows the sensitivity gap is a property of using a *single shared threshold*
  across groups with different score distributions -- it does not mean the underlying
  inequity disappears, only that a different (group-specific) operating point would
  equalize this particular criterion. Note that under equal-sensitivity thresholds the
  demographic-parity and disparate-impact gaps *widen* instead -- there is no single
  threshold choice that equalizes every criterion simultaneously.
- **Case-mix waterfall:** Adding demographic covariates (age, education, income,
  province) *increases* the male coefficient on the diabetes outcome (the raw gap is
  partly suppressed by these covariates being correlated with sex); comorbidities,
  behavioral factors, and access to care each modestly reduce it again, but none of the
  blocks come close to explaining away the sex gap in the outcome itself.
- **Sample size:** With n ~ 30,000 in the test set and ~3,000 positives, this is by far
  the largest and best-powered dataset in the project, giving tight confidence
  intervals on every estimate.

As with the other datasets, none of this should be read as showing the model is
"unbiased" or that fairness doesn't matter. The residual FNR gap -- present here at
high precision, and broadly consistent with three of the four other datasets -- is the
strongest evidence in this project for a real, not merely statistical, sex disparity in
how well diabetes-prediction models serve men vs. women.


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

fig1 = figs.plot_calibration_curve(
    result["y_test"], result["prob"], result["g_test"],
    group_labels={"Male": "Male", "Female": "Female"},
    title="Calibration by sex (calibrated model) -- CCHS 2019-2020 diabetes",
)
fig1.savefig(RESULTS_DIR / "05_cchs_calibration.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    gap_dict = {
        "Raw": (raw["point"], raw["ci_low"], raw["ci_high"]),
        "Prevalence-adjusted": (adjd["point"], adjd["ci_low"], adjd["ci_high"]),
    }
    fig = figs.plot_gap_bars(
        gap_dict,
        title=f"{key}: raw vs. prevalence-adjusted gap (Male - Female)",
        ylabel="Gap (Male - Female)",
    )
    fig.savefig(RESULTS_DIR / f"05_cchs_gap_{key}.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
fig_wf = figs.plot_case_mix_waterfall(
    result["case_mix"],
    title="CCHS 2019-2020: case-mix waterfall (Male vs Female diabetes gap)",
)
fig_wf.savefig(RESULTS_DIR / "05_cchs_case_mix_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


## Summary table (for cross-dataset pooling)

In [ ]:
rows = []
for key, boot in result["bootstrap_raw"].items():
    rows.append({
        "dataset": "cchs_diabetes",
        "metric": key,
        "raw_gap": boot["point"],
        "raw_ci_low": boot["ci_low"],
        "raw_ci_high": boot["ci_high"],
        "prevalence_adjusted_gap": None,
        "prevalence_adjusted_ci_low": None,
        "prevalence_adjusted_ci_high": None,
        "attenuation_pct": None,
    })

for key, boot in result["bootstrap_prevalence_adjusted"].items():
    raw = result["bootstrap_raw"][key]
    attenuation = None
    if raw["point"] != 0:
        attenuation = 100 * (1 - boot["point"] / raw["point"])
    for row in rows:
        if row["metric"] == key:
            row["prevalence_adjusted_gap"] = boot["point"]
            row["prevalence_adjusted_ci_low"] = boot["ci_low"]
            row["prevalence_adjusted_ci_high"] = boot["ci_high"]
            row["attenuation_pct"] = attenuation

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "05_cchs_summary.csv", index=False)
summary
